# 08c — Evaluación en test (split por bloques + normalización propia)

Evalúa los checkpoints de `07c_train_bloques_tuned.ipynb` (`v2_bloques_tuned`) contra el mismo
`manifest_test.csv` del split por bloques. Los loaders usan la misma normalización propia del 07c.

Salidas bajo `v2_bloques_tuned/`:

```text
data/08_reporting/v2_bloques_tuned/test_results_summary_v2_bloques_tuned.csv
data/08_reporting/v2_bloques_tuned/test_results_v2_bloques_tuned.txt
reports/figures/{arquitectura}_v2_bloques_tuned_test_confusion.png
reports/figures/v2_bloques_tuned_precision_recall.png
```


## 1. Configuración

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix, precision_recall_curve
from torch.utils.data import DataLoader

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.dataset import GarimpoDataset, load_manifest
from src.models.train import create_model
from src.utils.helpers import get_device, set_seed
from src.utils.metrics import compute_metrics

RUN_NAME = "v2_bloques_tuned"
SPLIT_NAME = "v2_bloques"

DATA_DIR = ROOT / "data"
MANIFEST_DIR = DATA_DIR / "05_model_input" / SPLIT_NAME
MODELS_DIR = DATA_DIR / "06_models" / RUN_NAME
REPORT_DIR = DATA_DIR / "08_reporting" / RUN_NAME
FIGURES_DIR = ROOT / "reports" / "figures"
MODELS_DIR = DATA_DIR / "06_models" / RUN_NAME
REPORT_DIR = DATA_DIR / "08_reporting" / RUN_NAME
FIGURES_DIR = ROOT / "reports" / "figures"

CHIPS_DIR_CANDIDATOS = [
    DATA_DIR / "01_raw" / "dataset_amazonia_garimpo_binario",
    DATA_DIR / "Dataset" / "datasets" / "amazonia_garimpo" / "dataset_amazonia_garimpo_binario",
]
CHIPS_DIR = next((p for p in CHIPS_DIR_CANDIDATOS if p.exists()), None)
if CHIPS_DIR is None:
    rutas = "\n  ".join(str(p) for p in CHIPS_DIR_CANDIDATOS)
    raise FileNotFoundError(f"No se encontró la carpeta de chips. Probadas:\n  {rutas}")

BATCH_SIZE = 32
NUM_WORKERS = 0
SEED = 42
LABELS = ["sem_garimpo", "com_garimpo"]
UMBRAL_BASE = 0.5
UMBRALES = np.round(np.arange(0.05, 0.96, 0.01), 2)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = get_device()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)

print(f"Corrida: {RUN_NAME}")
print(f"Device: {device}")
print(f"GPU: {gpu_name}")
print(f"PyTorch: {torch.__version__}")
print(f"Checkpoints : {MODELS_DIR}")
print(f"Manifiestos : {MANIFEST_DIR}")


## 2. DataLoaders de val y test

Mismos manifiestos que `v2_bloques` y la misma normalización propia que en `07c`. Val solo sirve
para calibrar el umbral; test da la métrica final.


In [ ]:
import json
from torchvision import transforms

NORM_JSON = DATA_DIR / "08_reporting" / "normalization_constants.json"
if not NORM_JSON.exists():
    raise FileNotFoundError(
        f"No se encontró {NORM_JSON}. Ejecuta antes 03b_normalizacion_chips.ipynb."
    )
_norm = json.loads(NORM_JSON.read_text(encoding="utf-8"))
DATASET_MEAN = tuple(_norm["DATASET_MEAN"])
DATASET_STD = tuple(_norm["DATASET_STD"])
print(f"Normalización propia (desde {NORM_JSON.name}):")
print(f"  DATASET_MEAN = {DATASET_MEAN}")
print(f"  DATASET_STD  = {DATASET_STD}")


def get_transforms_tuned(split: str) -> transforms.Compose:
    """Mismos augments que get_transforms(), con normalización medida sobre train."""
    normalize = transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
    if split == "train":
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=90),
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.1,
                hue=0.05,
            ),
            transforms.ToTensor(),
            normalize,
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ])


In [ ]:
def crear_loader(split: str) -> DataLoader:
    manifest = load_manifest(MANIFEST_DIR / f"manifest_{split}.csv")
    dataset = GarimpoDataset(
        manifest=manifest,
        chips_dir=CHIPS_DIR,
        transform=get_transforms_tuned(split),
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )


val_loader = crear_loader("val")
test_loader = crear_loader("test")

for nombre, loader in (("val", val_loader), ("test", test_loader)):
    manifest = loader.dataset.manifest
    print(
        f"{nombre:<5} {len(manifest):>7,} chips | "
        f"{len(loader):>5,} batches | "
        f"com_garimpo {manifest['label_int'].mean() * 100:5.1f} %"
    )


## 3. Checkpoints de la corrida

Se buscan solo dentro de `data/06_models/v2_bloques_tuned/`. Ejecutar antes `07c_train_bloques_tuned.ipynb`.


In [ ]:
checkpoint_paths = sorted(MODELS_DIR.rglob("*_best.pt"))

print(f"Checkpoints encontrados en {MODELS_DIR}: {len(checkpoint_paths)}")
for p in checkpoint_paths:
    print(f"  - {p.name}")

if not checkpoint_paths:
    print(
        "\nNo hay checkpoints de esta corrida.\n"
        "Ejecuta antes 07c_train_bloques_tuned.ipynb."
    )


## 4. Helpers de evaluación

`inferir()` devuelve probabilidades en lugar de predicciones ya decididas, que es lo que hace
falta para poder mover el umbral después. Con umbral 0.5 el resultado es idéntico al `argmax`
que usa el notebook 05, así que las métricas principales siguen siendo comparables.

In [ ]:
@torch.no_grad()
def inferir(ckpt_path: Path, loader: DataLoader, device) -> dict:
    """Carga un checkpoint y devuelve etiquetas reales y probabilidad de com_garimpo."""
    ckpt = torch.load(ckpt_path, map_location=device)
    model_name = ckpt["model_name"]
    img_size = ckpt.get("img_size", 128)

    model = create_model(model_name, img_size=img_size, pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    y_true, y_prob = [], []
    for images, labels in loader:
        images = images.to(device)
        probs = torch.softmax(model(images), dim=1)[:, 1]
        y_prob.extend(probs.cpu().tolist())
        y_true.extend(labels.tolist())

    return {
        "model_name": model_name,
        "train_best_epoch": ckpt.get("epoch"),
        "y_true": np.asarray(y_true),
        "y_prob": np.asarray(y_prob),
    }


def metricas_en_umbral(y_true: np.ndarray, y_prob: np.ndarray, umbral: float) -> dict:
    return compute_metrics(y_true, (y_prob >= umbral).astype(int))


def plot_confusion_matrix(y_true, y_pred, title: str, save_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABELS, yticklabels=LABELS, ax=ax,
    )
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_title(title)
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

## 5. Evaluar cada checkpoint en test

In [ ]:
results = []
inferencias = {}
log_lines = [
    f"EVALUACIÓN EN TEST ({RUN_NAME}) — RESULTADOS",
    f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"Device: {device} | GPU: {gpu_name}",
    f"Manifiestos: {MANIFEST_DIR}",
    f"Test chips: {len(test_loader.dataset):,} "
    f"(com_garimpo {test_loader.dataset.manifest['label_int'].mean() * 100:.1f} %)",
    f"Umbral de decisión: {UMBRAL_BASE}",
    "",
]

for i, ckpt_path in enumerate(checkpoint_paths, start=1):
    print("\n" + "=" * 60)
    print(f"[{i}/{len(checkpoint_paths)}] Evaluando: {ckpt_path.name}")
    print("=" * 60)

    inf = inferir(ckpt_path, test_loader, device)
    inferencias[ckpt_path] = inf

    m = metricas_en_umbral(inf["y_true"], inf["y_prob"], UMBRAL_BASE)
    label = f"{inf['model_name']} [{RUN_NAME}]"

    print(
        f"accuracy {m['accuracy']:.4f} | macro F1 {m['macro_f1']:.4f} | "
        f"recall com_garimpo {m['recall_com_garimpo']:.4f} | "
        f"precision com_garimpo {m['precision_com_garimpo']:.4f}"
    )

    plot_confusion_matrix(
        inf["y_true"],
        (inf["y_prob"] >= UMBRAL_BASE).astype(int),
        title=f"{label} — Test",
        save_path=FIGURES_DIR / f"{inf['model_name']}_{RUN_NAME}_test_confusion.png",
    )

    log_lines += [
        f"MODELO: {label} ({ckpt_path.name})",
        f"  test accuracy: {m['accuracy']:.4f}",
        f"  test macro F1: {m['macro_f1']:.4f}",
        f"  f1 com_garimpo: {m['f1_com_garimpo']:.4f} | f1 sem_garimpo: {m['f1_sem_garimpo']:.4f}",
        f"  recall com_garimpo: {m['recall_com_garimpo']:.4f} | precision com_garimpo: {m['precision_com_garimpo']:.4f}",
        "",
    ]

    results.append({
        "model": inf["model_name"],
        "variant": RUN_NAME,
        "checkpoint": ckpt_path.name,
        "train_best_epoch": inf["train_best_epoch"],
        "test_accuracy": round(m["accuracy"], 4),
        "test_macro_f1": round(m["macro_f1"], 4),
        "test_f1_com_garimpo": round(m["f1_com_garimpo"], 4),
        "test_f1_sem_garimpo": round(m["f1_sem_garimpo"], 4),
        "test_recall_com_garimpo": round(m["recall_com_garimpo"], 4),
        "test_precision_com_garimpo": round(m["precision_com_garimpo"], 4),
    })

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nEvaluación completada.")

## 6. Resumen comparativo en test

In [ ]:
results_df = pd.DataFrame(results).sort_values("test_macro_f1", ascending=False)
display(results_df)

summary_csv = REPORT_DIR / f"test_results_summary_{RUN_NAME}.csv"
results_df.to_csv(summary_csv, index=False)

best = results_df.iloc[0]
log_lines += [
    "=" * 60,
    f"RESUMEN COMPARATIVO (test, {RUN_NAME})",
    "=" * 60,
    "",
    results_df.to_string(index=False),
    "",
    f"MEJOR MODELO EN TEST: {best['model']} | "
    f"test macro F1 = {best['test_macro_f1']} | accuracy = {best['test_accuracy']}",
    f"Checkpoint: {best['checkpoint']}",
]

summary_txt = REPORT_DIR / f"test_results_{RUN_NAME}.txt"
summary_txt.write_text("\n".join(log_lines), encoding="utf-8")

print("\nArchivos guardados:")
print(f"  {summary_csv}")
print(f"  {summary_txt}")
print(f"\nMejor modelo en TEST: {best['model']}")
print(f"  Accuracy: {best['test_accuracy'] * 100:.1f} %")
print(f"  Macro F1: {best['test_macro_f1'] * 100:.1f} %")

## 7. Val vs. test

Con el split por bloques, val y test tienen prácticamente la misma prevalencia de garimpo, así
que la caída de macro F1 entre uno y otro ya sí mide sobreajuste y no un cambio de dificultad.
En la v1 esta comparación no era interpretable: test subía respecto a val simplemente porque
tenía un 19 % de positivos frente al 45 % de val.

In [ ]:
val_summary_csv = REPORT_DIR / f"training_results_summary_{RUN_NAME}.csv"

if val_summary_csv.exists():
    val_df = pd.read_csv(val_summary_csv)[["timm_name", "val_macro_f1", "val_accuracy"]]
    comparison = results_df.merge(
        val_df, left_on="model", right_on="timm_name", how="left"
    ).drop(columns=["timm_name"])
    comparison["f1_drop_val_to_test"] = (
        comparison["val_macro_f1"] - comparison["test_macro_f1"]
    ).round(4)

    cols = [
        "model", "val_macro_f1", "test_macro_f1", "f1_drop_val_to_test",
        "val_accuracy", "test_accuracy",
    ]
    display(comparison[cols].sort_values("f1_drop_val_to_test"))

    comparison_csv = REPORT_DIR / f"val_vs_test_comparison_{RUN_NAME}.csv"
    comparison[cols].to_csv(comparison_csv, index=False)
    print(f"\nGuardado: {comparison_csv}")
else:
    print(f"No se encontró {val_summary_csv} (corrida de 07_train_bloques.ipynb).")

## 8. Calibración del umbral de decisión

El umbral 0.5 no tiene nada de especial: es lo que sale por defecto de un `argmax`. Aquí se
barre el umbral **sobre validación**, se elige el que maximiza el macro F1, y ese umbral se
aplica a test. Como la elección se hace en val, la métrica de test sigue siendo honesta.

Esto ataca directamente el punto débil de la v1: la precisión de `com_garimpo`. Subir el umbral
reduce falsas alarmas a costa de recall; bajarlo hace lo contrario. La tabla deja ver el
intercambio con números en lugar de en abstracto.

In [ ]:
filas_umbral = []

for ckpt_path in checkpoint_paths:
    inf_test = inferencias[ckpt_path]
    inf_val = inferir(ckpt_path, val_loader, device)
    nombre = inf_test["model_name"]

    barrido = [
        (u, metricas_en_umbral(inf_val["y_true"], inf_val["y_prob"], u)["macro_f1"])
        for u in UMBRALES
    ]
    umbral_opt = max(barrido, key=lambda par: par[1])[0]

    base = metricas_en_umbral(inf_test["y_true"], inf_test["y_prob"], UMBRAL_BASE)
    calibrado = metricas_en_umbral(inf_test["y_true"], inf_test["y_prob"], umbral_opt)

    filas_umbral.append({
        "model": nombre,
        "umbral_optimo_val": umbral_opt,
        "test_macro_f1_05": round(base["macro_f1"], 4),
        "test_macro_f1_calibrado": round(calibrado["macro_f1"], 4),
        "ganancia_f1": round(calibrado["macro_f1"] - base["macro_f1"], 4),
        "precision_com_05": round(base["precision_com_garimpo"], 4),
        "precision_com_calibrado": round(calibrado["precision_com_garimpo"], 4),
        "recall_com_05": round(base["recall_com_garimpo"], 4),
        "recall_com_calibrado": round(calibrado["recall_com_garimpo"], 4),
    })

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

umbral_df = pd.DataFrame(filas_umbral).sort_values("test_macro_f1_calibrado", ascending=False)
umbral_df.to_csv(REPORT_DIR / f"umbral_calibrado_{RUN_NAME}.csv", index=False)
display(umbral_df)

print(f"\nGuardado: {REPORT_DIR / f'umbral_calibrado_{RUN_NAME}.csv'}")

### Curvas precisión–recall en test

Cada curva muestra todos los puntos de operación posibles de un modelo. Es la figura que permite
justificar el umbral elegido según el uso: detección exhaustiva (recall alto) o mapa operativo
con pocas falsas alarmas (precisión alta).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for ckpt_path in checkpoint_paths:
    inf = inferencias[ckpt_path]
    precision, recall, _ = precision_recall_curve(inf["y_true"], inf["y_prob"])
    ax.plot(recall, precision, lw=1.8, label=inf["model_name"])

prevalencia = test_loader.dataset.manifest["label_int"].mean()
ax.axhline(prevalencia, color="black", ls="--", lw=1, label=f"azar ({prevalencia:.2f})")

ax.set_xlabel("Recall com_garimpo")
ax.set_ylabel("Precisión com_garimpo")
ax.set_title(f"Curvas precisión–recall en test ({RUN_NAME})")
ax.legend(loc="lower left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f"{RUN_NAME}_precision_recall.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Referencia: resultados de la v1

**Los dos conjuntos de test son distintos**, así que estas cifras no se pueden comparar
directamente: el test de la v1 tenía un 19 % de positivos y el de la v2 ronda el 50 %, lo que
hace que la accuracy de la v1 se vea inflada por la clase mayoritaria. La tabla sirve para
documentar ambas corridas en el informe, no para declarar un ganador entre ellas.

In [ ]:
baseline_csv = DATA_DIR / "08_reporting" / "v2_bloques" / "test_results_summary_v2_bloques.csv"

if baseline_csv.exists():
    base_df = pd.read_csv(baseline_csv)[
        ["model", "test_macro_f1", "test_precision_com_garimpo", "test_recall_com_garimpo"]
    ].rename(columns={
        "test_macro_f1": "test_macro_f1_v2_bloques",
        "test_precision_com_garimpo": "precision_com_v2_bloques",
        "test_recall_com_garimpo": "recall_com_v2_bloques",
    })
    tuned_df = results_df[[
        "model", "test_macro_f1", "test_precision_com_garimpo", "test_recall_com_garimpo",
    ]].rename(columns={
        "test_macro_f1": "test_macro_f1_v2_bloques_tuned",
        "test_precision_com_garimpo": "precision_com_v2_tuned",
        "test_recall_com_garimpo": "recall_com_v2_tuned",
    })
    comparacion = base_df.merge(tuned_df, on="model", how="outer")
    comparacion["f1_delta_tuned_menos_baseline"] = (
        comparacion["test_macro_f1_v2_bloques_tuned"] - comparacion["test_macro_f1_v2_bloques"]
    ).round(4)
    display(comparacion.sort_values("f1_delta_tuned_menos_baseline", ascending=False))

    compare_csv = REPORT_DIR / f"v2_bloques_vs_{RUN_NAME}_test_comparison.csv"
    comparacion.to_csv(compare_csv, index=False)
    print(f"\nGuardado: {compare_csv}")
    print("\nMismo test set (~50/50): la comparación es directamente interpretable.")
else:
    print(f"No se encontró {baseline_csv}. Ejecuta antes 08_eval_bloques.ipynb.")


## 10. Qué llevarse al informe

De esta corrida salen las tres piezas que faltaban:

- **Comparación limpia de 4 arquitecturas** bajo un protocolo único y con val y test
  equilibrados, así que el modelo elegido en validación es el que debería ganar en test.
- **Punto de operación justificado**, con el umbral elegido en val y las curvas precisión–recall
  para defender la decisión.
- **Ablaciones de la v1** (preentrenado vs. scratch, LR original vs. ajustado, split por ráster
  vs. por bloques) que siguen intactas en `data/08_reporting/`.